In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
from nemosis import static_table
from datetime import timedelta
import dask.dataframe as dd
from concurrent.futures import ProcessPoolExecutor
import time

raw_data_cache = '/Volumes/T7/NEMO-misc'

In [9]:
# Join DUIDs with firm names and further info
generator_info_df = static_table(table_name='Generators and Scheduled Loads', 
                              raw_data_location=raw_data_cache,
                              update_static_file=False)
generator_info_df

INFO: Retrieving static table Generators and Scheduled Loads


,Participant,Station Name,Region,Dispatch Type,Category,Classification,Fuel Source - Primary,Fuel Source - Descriptor,Technology Type - Primary,Technology Type - Descriptor,Aggregation,DUID
0,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,ADPBA1G
1,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Load,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,ADPBA1L
2,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Hydro,Water,Renewable,Run of River,Y,ADPMH1
3,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Semi-Scheduled,Solar,Solar,Renewable,Photovoltaic Tracking Flat panel,Y,ADPPV1
4,South Australian Water Corporation,Adelaide Desalination Plant,SA1,Generating Unit,Market,Non-Scheduled,Solar,Solar,Renewable,Photovoltaic Flat panel,Y,ADPPV2
...,...,...,...,...,...,...,...,...,...,...,...,...
527,Tailem Bend II Project Company Pty Ltd as trus...,Tailem Bend 2 Hybrid Renewable Power Station,SA1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,TB2B1
528,AGL Macquarie Pty Limited,Broken Hill Battery Energy Storage System,NSW1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,BHB1
529,AGL SA Generation Pty Limited,Torrens Island BESS,SA1,Bidirectional Unit,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,TIB1
530,Capital Battery Pty Ltd as Trustee for Capital...,Capital Battery,NSW1,Load,Market,Scheduled,Battery storage,Grid,Storage,Battery and Inverter,Y,CAPBES1


In [ ]:
parquet_dir = "/Volumes/T7/bid-volume-data-parquet-sorted"

all_files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
# Exclude any files starting with '._'
valid_files = [f for f in all_files if not os.path.basename(f).startswith("._")]

ddf = dd.read_parquet(valid_files)
print(ddf.head())

        SETTLEMENTDATE    DUID    BIDTYPE            OFFERDATE MAXAVAIL  \
0  2009/07/01 00:00:00  AGLHAL     ENERGY  2009/06/19 14:55:14      183   
1  2009/07/01 00:00:00  AGLSOM     ENERGY  2009/06/30 11:23:49      150   
2  2009/07/01 00:00:00  ANGAS1     ENERGY  2009/06/30 11:23:51       30   
3  2009/07/01 00:00:00  ANGAS2     ENERGY  2009/06/30 11:23:51       20   
4  2009/07/01 00:00:00   APD01  LOWER5MIN  2002/10/17 08:51:56        0   

  ENABLEMENTMIN ENABLEMENTMAX LOWBREAKPOINT HIGHBREAKPOINT BANDAVAIL1  \
0             0             0             0              0          0   
1             0             0             0              0          0   
2             0             0             0              0          0   
3             0             0             0              0          0   
4             0             0             0              0          0   

  BANDAVAIL2 BANDAVAIL3 BANDAVAIL4 BANDAVAIL5 BANDAVAIL6 BANDAVAIL7  \
0          0          0          0     

In [10]:
# Directories
input_dir = "/Volumes/T7/bid-volume-data-parquet-sorted"

# Output directories
merged_output_dir = "/Volumes/T7/enriched_merged"
os.makedirs(merged_output_dir, exist_ok=True)

# 1) Load generator_info_df into a small pandas or Dask DataFrame
#    If generator_info_df is already in memory as a pandas DF, skip this read.
import pandas as pd
generator_info_ddf = dd.from_pandas(generator_info_df, npartitions=1)

# 2) Collect all Parquet files, skipping hidden ones
parquet_files = sorted(
    f for f in glob.glob(os.path.join(input_dir, "*.parquet"))
    if not os.path.basename(f).startswith("._")
)

print(f"Found {len(parquet_files)} parquet files in {input_dir}.")

# 3) Read them all into a single Dask DataFrame
ddf = dd.read_parquet(parquet_files)
print("Dask DataFrame loaded. Now merging with generator_info_df...")

# 4) Merge on 'DUID'
enriched_ddf = ddf.merge(generator_info_ddf, on='DUID', how='left')

# 5) Write out the merged dataset (still a Dask DF)
print(f"Writing merged data to {merged_output_dir} ...")
enriched_ddf.to_parquet(merged_output_dir, overwrite=True)
print("Merge complete!")

Found 1448 parquet files in /Volumes/T7/bid-volume-data-parquet-sorted.
Dask DataFrame loaded. Now merging with generator_info_df...
Writing merged data to /Volumes/T7/enriched_merged ...
Merge complete!


In [ ]:
output_dir = '/Volumes/T7/enriched_sa1_sample/filtered_sample.parquet/part.0.parquet'

ddf = dd.read_parquet(output_dir)
print("Sample file loaded.")

# Step 2: Inspect the data
print("Data types:\n", ddf.dtypes)
print("First few rows:\n", ddf.head())

Sample file loaded.
Data types:
 SETTLEMENTDATE                  object
DUID                            object
BIDTYPE                         object
OFFERDATE                       object
MAXAVAIL                        object
ENABLEMENTMIN                   object
ENABLEMENTMAX                   object
LOWBREAKPOINT                   object
HIGHBREAKPOINT                  object
BANDAVAIL1                      object
BANDAVAIL2                      object
BANDAVAIL3                      object
BANDAVAIL4                      object
BANDAVAIL5                      object
BANDAVAIL6                      object
BANDAVAIL7                      object
BANDAVAIL8                      object
BANDAVAIL9                      object
BANDAVAIL10                     object
INTERVAL_DATETIME               object
Participant                     object
Station Name                    object
Region                          object
Dispatch Type                   object
Category                       